In [ ]:
import pandas as pd
import numpy as np

# Create a Data Ingestion Function

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris

def data_ingestion(filename):
    """
    Load the dataset and return it as a pandas DataFrame.
    """

    # Reading the data
    df = pd.read_csv(filename)

    return df

In [ ]:
filename = "/content/student_dataset_50_samples_missing.csv"
df = data_ingestion(filename)
df.head()

,Student_ID,Age,Gender,Study_Hours,Attendance,Assignment_Score,Final_Score,Result
0,S1001,24.0,Female,4.6,88.0,NaN,35.0,Fail
1,S1002,21.0,Female,7.7,95.0,41.0,45.0,Pass
2,S1003,28.0,Male,6.9,72.0,92.0,91.0,Pass
3,S1004,25.0,Female,6.2,91.0,83.0,84.0,Pass
4,S1005,22.0,Female,4.8,66.0,65.0,57.0,Pass


## Create a Data Validation Function

In [ ]:
def data_validation(df):
    """
    Validate the dataset before training.
    """

    print("=" * 50)
    print("DATA VALIDATION REPORT")
    print("=" * 50)

    # Shape
    print(f"Dataset Shape : {df.shape}")

    # Missing Values
    print("\nMissing Values:")
    print(df.isnull().sum())

    # Duplicate Rows
    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    # Data Types
    print("\nData Types:")
    print(df.dtypes)

    # Target Column
    if "target" in df.columns:
        print("\nTarget column found ✔")
    else:
        print("\nTarget column missing ❌")

    print("=" * 50)

    return df

In [ ]:
filename = "/content/student_dataset_50_samples_missing.csv"
df = data_ingestion(filename)
df = data_validation(df)

DATA VALIDATION REPORT
Dataset Shape : (50, 8)

Missing Values:
Student_ID          0
Age                 3
Gender              0
Study_Hours         3
Attendance          3
Assignment_Score    3
Final_Score         3
Result              0
dtype: int64

Duplicate Rows:
0

Data Types:
Student_ID           object
Age                 float64
Gender               object
Study_Hours         float64
Attendance          float64
Assignment_Score    float64
Final_Score         float64
Result               object
dtype: object

Target column missing ❌


In [ ]:
from sklearn.model_selection import train_test_split

def split_data(df):


    y = df[["Result"]]
    X = df.drop("Result", axis=1)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    print("Train Test Split Completed")

    return X_train, X_test, y_train, y_test

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

def data_transformation(X_train, X_test):

    numerical_columns = X_train.select_dtypes(
        include=["int64", "float64"]
    ).columns

    categorical_columns = X_train.select_dtypes(
        include=["object"]
    ).columns

    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numerical_pipeline, numerical_columns),
        ("cat", categorical_pipeline, categorical_columns)
    ])

    X_train = preprocessor.fit_transform(X_train)

    X_test = preprocessor.transform(X_test)

    print("Data Transformation Completed")

    return X_train, X_test, preprocessor

In [ ]:
filename = "/content/student_dataset_50_samples_missing.csv"
df = data_ingestion(filename)
df = data_validation(df)
X_train, X_test, y_train, y_test = split_data(df)

X_train, X_test, preprocessor = data_transformation(
    X_train,
    X_test
)


DATA VALIDATION REPORT
Dataset Shape : (50, 8)

Missing Values:
Student_ID          0
Age                 3
Gender              0
Study_Hours         3
Attendance          3
Assignment_Score    3
Final_Score         3
Result              0
dtype: int64

Duplicate Rows:
0

Data Types:
Student_ID           object
Age                 float64
Gender               object
Study_Hours         float64
Attendance          float64
Assignment_Score    float64
Final_Score         float64
Result               object
dtype: object

Target column missing ❌
Train Test Split Completed
Data Transformation Completed


## Create Model Training Function

In [ ]:
from sklearn.tree import DecisionTreeClassifier

def model_training(X_train, y_train):
    """
    Train the Decision Tree model.
    """

    # Create model
    model = DecisionTreeClassifier(
        criterion="gini",
        max_depth=3,
        random_state=42
    )

    # Train model
    model.fit(X_train, y_train)

    print("="*50)
    print("MODEL TRAINING COMPLETED")
    print("="*50)

    return model

In [ ]:
model = model_training(X_train, y_train)

MODEL TRAINING COMPLETED


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

def model_evaluation(model, X_test, y_test):
    """
    Evaluate the trained model.
    """

    # Prediction
    y_pred = model.predict(X_test)

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted")
    recall = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")

    print("=" * 50)
    print("MODEL EVALUATION")
    print("=" * 50)

    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")

    print("\nConfusion Matrix")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report")
    print(classification_report(y_test, y_pred))

    return accuracy, precision, recall, f1

In [ ]:
accuracy, precision, recall, f1 = model_evaluation(
    model,
    X_test,
    y_test
)

MODEL EVALUATION
Accuracy  : 0.5000
Precision : 0.4375
Recall    : 0.5000
F1 Score  : 0.4667

Confusion Matrix
[[0 3]
 [2 5]]

Classification Report
              precision    recall  f1-score   support

        Fail       0.00      0.00      0.00         3
        Pass       0.62      0.71      0.67         7

    accuracy                           0.50        10
   macro avg       0.31      0.36      0.33        10
weighted avg       0.44      0.50      0.47        10

